# 00_02 — Validación del Modelo PSBP

**Modelo:** Probit Stick-Breaking Process (Chung & Dunson, 2009)  
**Tipo de datos:** Simulados  
**Notebook:** autocontenido — `run_psbp` definida localmente  

---

### Rutas del proyecto

| Rol | Ruta |
|-----|------|
| Datos de entrada (train + test) | `data/simulaciones/raw/` |
| Predicciones generadas | `data/simulaciones/processed/predict/` |
| Artefacto del modelo | `artefact/simulaciones/models/` |
| Reportes | `reports/simulaciones/` |

## 0. Imports y configuración de rutas

In [ ]:
import numpy as np
import pickle
import json
import os
import sys
from pathlib import Path
from datetime import datetime
from scipy.stats import norm
from scipy.special import gammaln

# ── Raíz del proyecto (para Jupyter notebooks) ──────────────────────────────
def get_project_root(marker="README.md"):
    """Encuentra la raíz del proyecto buscando un archivo/carpeta marca"""
    # En Jupyter, usamos el directorio actual de trabajo
    current = Path(os.getcwd()).resolve()
    print(f" Buscando desde: {current}")
    
    # Buscar hacia arriba hasta encontrar el marker
    for parent in [current] + list(current.parents):
        if (parent / marker).exists():
            print(f"✓ Marcador '{marker}' encontrado en: {parent}")
            return parent
    
    print(f" No se encontró '{marker}', usando directorio actual")
    return current

# Buscar raíz del proyecto
PROJECT_ROOT = get_project_root("README.md")
print(f" PROJECT_ROOT: {PROJECT_ROOT}")

# ── Rutas canónicas del proyecto ─────────────────────────────────────────────
PATHS = {
    "raw":       PROJECT_ROOT / "data" / "simulaciones" / "raw",
    "predict":   PROJECT_ROOT / "data" / "simulaciones" / "processed" / "predict",
    "artifact":  PROJECT_ROOT / "artefact" / "simulaciones",
    "reports":   PROJECT_ROOT / "reports" / "simulaciones",
}

# Crear directorios si no existen y listar
for name, path in PATHS.items():
    path.mkdir(parents=True, exist_ok=True)
    print(f"  {name:10s} → {path}")

 Buscando desde: C:\Users\JuanFran\Desktop\git_tesis\model_psbp_fd\notebooks\simulaciones
✓ Marcador 'README.md' encontrado en: C:\Users\JuanFran\Desktop\git_tesis\model_psbp_fd
 PROJECT_ROOT: C:\Users\JuanFran\Desktop\git_tesis\model_psbp_fd
  raw        → C:\Users\JuanFran\Desktop\git_tesis\model_psbp_fd\data\simulaciones\raw
  predict    → C:\Users\JuanFran\Desktop\git_tesis\model_psbp_fd\data\simulaciones\processed\predict
  artifact   → C:\Users\JuanFran\Desktop\git_tesis\model_psbp_fd\artefact\simulaciones\models
  reports    → C:\Users\JuanFran\Desktop\git_tesis\model_psbp_fd\reports\simulaciones


## 1. Configuración del experimento

Todos los parámetros editables están centralizados aquí.  
El `experiment_id` se usa para nombrar todos los artefactos de salida.

In [11]:
# ── Identificación del experimento ───────────────────────────────────────────
BASEFNAME    = "BHP"          # prefijo de los archivos de datos
TT           = 1              # índice del experimento/fold
SEED         = 42             # semilla para reproducibilidad
TIMESTAMP    = datetime.now().strftime("%Y%m%d_%H%M%S")
EXPERIMENT_ID = f"psbp_sim_{BASEFNAME}_{TT}_{TIMESTAMP}"

print(f"Experiment ID : {EXPERIMENT_ID}")
print(f"Seed          : {SEED}")

# ── Hiperparámetros MCMC ─────────────────────────────────────────────────────
MCMC_CONFIG = {
    "nsim": 2000,
    "burn": 200,
    "N":    20,       # truncamiento del proceso
    "M":    50,       # puntos de grilla para Gamma_jh
}

# ── Hiperparámetros del modelo ───────────────────────────────────────────────
HYPERPARAMS = {
    "atau":  0.5,
    "btau":  0.5,
    "ag":    0.5,
    "bg":    0.5,
    "apij":  1.0,    # escalar; se expande a (p,) dentro del modelo
    "bpij":  5.0,    # escalar; se expande a (p,)
    "mumu":  0.0,
    "taumu": 1.0,
    "mupsij":  0.0,  # escalar; se expande a (p,)
    "taupsij": 1.0,  # escalar; se expande a (p,)
    "pwj":   0.5,
}

print("\nMCMC config:")
for k, v in MCMC_CONFIG.items():
    print(f"  {k}: {v}")
print("\nHiperparámetros:")
for k, v in HYPERPARAMS.items():
    print(f"  {k}: {v}")

Experiment ID : psbp_sim_BHP_1_20260509_203336
Seed          : 42

MCMC config:
  nsim: 2000
  burn: 200
  N: 20
  M: 50

Hiperparámetros:
  atau: 0.5
  btau: 0.5
  ag: 0.5
  bg: 0.5
  apij: 1.0
  bpij: 5.0
  mumu: 0.0
  taumu: 1.0
  mupsij: 0.0
  taupsij: 1.0
  pwj: 0.5


## 2. Definición del modelo (`run_psbp`)

Implementación autocontenida. La función acepta rutas explícitas y los
diccionarios de configuración definidos en la celda anterior.

In [10]:
# ── Utilidades de muestreo ────────────────────────────────────────────────────

def truncnorm_lower0_sample(m, v, rng):
    """N(m,v) truncada en (-inf, 0]. Equivale a norminv(u*normcdf(...)) de MATLAB."""
    sv = np.sqrt(v)
    u = rng.uniform(0, 1)
    cdf0 = norm.cdf((0 - m) / sv)
    return m + sv * norm.ppf(u * cdf0)


def truncnorm_upper0_sample(m, v, rng):
    """N(m,v) truncada en [0, +inf). Equivale a norminv(u+(1-u)*normcdf(...)) de MATLAB."""
    sv = np.sqrt(v)
    u = rng.uniform(0, 1)
    cdf0 = norm.cdf((0 - m) / sv)
    val = m + sv * norm.ppf(u + (1 - u) * cdf0)
    return val if not (np.isinf(val) or np.isnan(val)) else 0.01


def mvnrnd(mu, cov, rng):
    """Normal multivariada. Equivale a mvnrnd de MATLAB."""
    return rng.multivariate_normal(mu.ravel(), cov)


# ── Función principal ─────────────────────────────────────────────────────────

def run_psbp(path_in: Path, path_out: Path, mcmc_cfg: dict, hp: dict, seed=None):
    """
    Ejecuta el Gibbs sampler del modelo PSBP.

    Parámetros
    ----------
    path_in  : Path al archivo de entrenamiento  (*.txt)
    path_out : Path al archivo de prueba          (*.txt)
    mcmc_cfg : dict con nsim, burn, N, M
    hp       : dict con hiperparámetros del modelo
    seed     : int opcional para reproducibilidad

    Retorna
    -------
    dict con todas las trazas MCMC y métricas post-burn-in.
    """
    rng = np.random.default_rng(seed)
    EPS = np.finfo(float).tiny

    # ── 1. Carga de datos ────────────────────────────────────────────────────
    dt  = np.loadtxt(path_in)
    dt2 = np.loadtxt(path_out)
    n, n2 = dt.shape[0], dt2.shape[0]
    dt1 = dt.copy()
    y2  = dt2[:, 0]

    # ── 2. Estandarización (estadísticas del train set) ──────────────────────
    dtmean = dt.mean(axis=0)
    dtstd  = dt.std(axis=0, ddof=0)   # ddof=0 ↔ std(dt,1) en MATLAB
    dt  = (dt  - dtmean) / dtstd
    dt2 = (dt2 - dtmean) / dtstd

    y      = dt[:, 0]
    Xnoint = dt[:, 1:]
    y1     = dt1[:, 0]
    n      = len(y)
    X      = np.hstack([np.ones((n, 1)), Xnoint])
    p      = Xnoint.shape[1]
    outX   = np.hstack([np.ones((n2, 1)), dt2[:, 1:]])
    outy   = dt2[:, 0]

    # ── 3. Parámetros MCMC y modelo ──────────────────────────────────────────
    nsim, burn = mcmc_cfg["nsim"], mcmc_cfg["burn"]
    N,    M    = mcmc_cfg["N"],    mcmc_cfg["M"]
    ncur, nrun = 1, nsim - 1

    atau, btau   = hp["atau"], hp["btau"]
    ag,   bg     = hp["ag"],   hp["bg"]
    apij  = np.full(p, hp["apij"])
    bpij  = np.full(p, hp["bpij"])
    mumu, taumu  = hp["mumu"], hp["taumu"]
    mupsij  = np.full(p, hp["mupsij"])
    taupsij = np.full(p, hp["taupsij"])
    pwj     = hp["pwj"]

    # ── 4. Inicialización ────────────────────────────────────────────────────
    mu  = 1.0
    g   = 1.0
    pij = 0.5 * np.ones(p)
    wj  = np.ones(p)

    xmin, xmax = X.min(), X.max()
    Gstar   = xmin + (np.arange(1, M + 1) / M) * (xmax - xmin)

    bjrange = np.array([-4, -3, -2, -1.5, -1, 0, 1, 1.5, 2, 3, 4, 5])
    betajh  = bjrange[rng.integers(0, 7, size=(N, p))]
    beta0h  = bjrange[rng.integers(0, 7, size=(N, 1))]
    tauh    = rng.gamma(shape=atau, scale=1 / btau, size=(N, 1))
    Si      = rng.integers(0, N, size=n)
    alphah  = np.zeros(N - 1)
    psijh   = np.zeros((N - 1, p))
    gammajh = np.ones((N, p))
    Gammajh = Gstar[rng.integers(0, M, size=(N - 1, p))]

    # ── Contenedores ─────────────────────────────────────────────────────────
    betajhout  = np.zeros((nsim, N, p),      dtype=np.float32)
    beta0hout  = np.zeros((nsim, N),         dtype=np.float32)
    tauhout    = np.zeros((nsim, N),         dtype=np.float32)
    alphahout  = np.zeros((nsim, N - 1),     dtype=np.float32)
    Gammajhout = np.zeros((nsim, N - 1, p),  dtype=np.float32)
    psijhout   = np.zeros((nsim, N - 1, p),  dtype=np.float32)
    gammajhout = np.zeros((nsim, N, p),      dtype=np.float32)
    pijout     = np.zeros((nsim, p),         dtype=np.float32)
    wjout      = np.zeros((nsim, p),         dtype=np.float32)
    N1out      = np.zeros(nsim,              dtype=np.float32)
    Nout       = np.zeros(nsim,              dtype=np.float32)
    muout      = np.zeros(nsim,              dtype=np.float32)
    osumout    = np.zeros((nsim, p),         dtype=np.float32)
    inEout     = np.zeros((nsim, n),         dtype=np.float32)
    outEout    = np.zeros((nsim, n2),        dtype=np.float32)

    # ── 5. Loop MCMC (Gibbs) ─────────────────────────────────────────────────
    for gt in range(ncur - 1, ncur - 1 + nrun + 1):

        # PASO 1 — Z_il (variables latentes del probit stick-breaking)
        Zil = np.zeros((n, N))
        Wil = np.zeros((n, N))
        for i in range(n):
            si  = Si[i]
            lim = si if si < N - 1 else N - 1
            for l in range(lim):
                m_l = alphah[l] - np.sum(psijh[l] * np.abs(Xnoint[i] - Gammajh[l]))
                if si < N - 1:
                    Zil[i, l] = (truncnorm_lower0_sample(m_l, 1.0, rng) if l < si
                                 else truncnorm_upper0_sample(m_l, 1.0, rng))
                else:
                    Zil[i, l] = truncnorm_lower0_sample(m_l, 1.0, rng)
                Wil[i, l] = Zil[i, l] + np.sum(psijh[l] * np.abs(X[i, 1:] - Gammajh[l]))

        # PASO 2 — S_i (asignaciones de componente)
        phxi    = np.zeros((n,  N))
        phxiout = np.zeros((n2, N))

        for i in range(n):
            vhx = np.array([norm.cdf(alphah[h] - np.sum(psijh[h] * np.abs(X[i, 1:] - Gammajh[h])))
                            for h in range(N - 1)])
            phx = np.zeros(N)
            for h in range(N - 1):
                phx[h] = vhx[h] if h == 0 else vhx[h] * np.prod(1 - vhx[:h])
            phx[N - 1] = np.prod(1 - vhx)
            phxi[i] = phx

            mu_h  = beta0h[:, 0] * X[i, 0] + betajh @ X[i, 1:]
            sig_h = 1.0 / np.sqrt(tauh[:, 0])
            lik   = norm.pdf(y[i], loc=mu_h, scale=sig_h)
            w     = (phx + EPS) * (lik + EPS)
            Si[i] = rng.choice(N, p=w / w.sum())

        for i in range(n2):
            vhx2 = np.array([norm.cdf(alphah[h] - np.sum(psijh[h] * np.abs(outX[i, 1:] - Gammajh[h])))
                             for h in range(N - 1)])
            phx2 = np.zeros(N)
            for h in range(N - 1):
                phx2[h] = vhx2[h] if h == 0 else vhx2[h] * np.prod(1 - vhx2[:h])
            phx2[N - 1] = np.prod(1 - vhx2)
            phxiout[i]  = phx2

        # PASO 3 — Predicciones esperadas por componente
        inE  = np.zeros((n,  N))
        outE = np.zeros((n2, N))
        for h in range(N):
            inE[:,  h] = phxi[:,    h] * (X[:,    0] * beta0h[h, 0] + X[:,    1:] @ betajh[h])
            outE[:, h] = phxiout[:, h] * (outX[:, 0] * beta0h[h, 0] + outX[:, 1:] @ betajh[h])

        # PASO 4 — beta_h (posterior conjugada Normal multivariada)
        betajh = np.zeros((N, p))
        for h in range(N):
            active = np.concatenate([[True], gammajh[h] == 1])
            Xh  = X[:, active]
            pgh = Xh.shape[1]
            idx = (Si == h)
            Sh     = (n / g) * np.linalg.inv(Xh.T @ Xh) / tauh[h, 0]
            Sh_inv = np.linalg.inv(Sh)
            if idx.sum() > 0:
                XhSi   = Xh[idx]
                Shhat  = np.linalg.inv(Sh_inv + tauh[h, 0] * XhSi.T @ XhSi)
                Shhat  = 0.5 * (Shhat + Shhat.T)
                muhhat = Shhat @ (tauh[h, 0] * XhSi.T @ y[idx])
            else:
                Shhat  = np.linalg.inv(Sh_inv)
                Shhat  = 0.5 * (Shhat + Shhat.T)
                muhhat = np.zeros(pgh)
            bt = mvnrnd(muhhat, Shhat, rng)
            beta0h[h, 0] = bt[0]
            cnt = 1
            for j in range(p):
                if gammajh[h, j] == 1:
                    betajh[h, j] = bt[cnt]; cnt += 1

        # PASO 5 — tau_h (posterior conjugada Gamma)
        for h in range(N):
            active = np.concatenate([[True], gammajh[h] == 1])
            Xh     = X[:, active]
            betagh = np.concatenate([[beta0h[h, 0]], betajh[h, gammajh[h] == 1]])
            idx    = (Si == h)
            rss    = ((y[idx] - Xh[idx] @ betagh) ** 2).sum() if idx.sum() > 0 else 0.0
            aa = atau + 0.5 * idx.sum() + 0.5 * gammajh[h].sum() + 0.5
            bb = btau + 0.5 * rss + 0.5 / n * g * betagh @ (Xh.T @ Xh) @ betagh
            tauh[h, 0] = rng.gamma(shape=aa, scale=1 / bb)

        # PASO 6 — g (escala global de prior de Zellner)
        aghat = ag + 0.5 * (gammajh.sum() + N)
        temp  = sum(
            tauh[h, 0] * (bg_vec := np.concatenate([[beta0h[h, 0]], betajh[h, gammajh[h] == 1]])) @
            (X[:, np.concatenate([[True], gammajh[h] == 1])].T @
             X[:, np.concatenate([[True], gammajh[h] == 1])]) @ bg_vec
            for h in range(N)
        )
        bghat = bg + 0.5 / n * temp
        g = rng.gamma(shape=aghat, scale=1 / bghat)

        # PASO 7 — w_j (indicador global de relevancia)
        for j in range(p):
            if gammajh[:, j].sum() > 0:
                wj[j] = 1
            else:
                b = np.exp(gammaln(bpij[j] + N) + gammaln(apij[j] + bpij[j])
                           - gammaln(bpij[j]) - gammaln(apij[j] + bpij[j] + N))
                pwjhat = pwj * b / ((1 - pwj) + pwj * b)
                wj[j]  = rng.binomial(1, pwjhat)

        # PASO 8 — alpha_h (posterior conjugada Normal)
        for h in range(N - 1):
            mask  = (Si >= h)
            v_ah  = 1.0 / (1 + mask.sum())
            m_ah  = v_ah * (mu + Wil[mask, h].sum())
            alphah[h] = rng.normal(m_ah, np.sqrt(v_ah))

        # PASO 9 — mu (posterior conjugada Normal)
        taumuhat = N + taumu
        mumuhat  = (taumu * mumu + alphah.sum()) / taumuhat
        mu = rng.normal(mumuhat, np.sqrt(1 / taumuhat))

        # PASO 10 — pi_j (posterior conjugada Beta)
        for j in range(p):
            if wj[j] == 0:
                pij[j] = 0.0
            else:
                pij[j] = rng.beta(apij[j] + gammajh[:, j].sum(),
                                   bpij[j] + N - gammajh[:, j].sum())

        # PASO 11 — Gamma_jh (localización de referencia)
        for h in range(N - 1):
            mask_h = (Si >= h)
            kh = mask_h.sum()
            for j in range(p):
                if gammajh[h, j] == 1 and kh > 0:
                    pm = np.zeros(M)
                    for m_idx in range(M):
                        od = (np.sum(psijh[h, :j]  * np.abs(Xnoint[mask_h, :j]  - Gammajh[h, :j]),  axis=1) +
                              np.sum(psijh[h, j+1:] * np.abs(Xnoint[mask_h, j+1:] - Gammajh[h, j+1:]), axis=1))
                        mz = alphah[h] - od - psijh[h, j] * np.abs(Xnoint[mask_h, j] - Gstar[m_idx])
                        pm[m_idx] = np.exp(1.2 * kh + norm.logpdf(Zil[mask_h, h], mz, 1.0).sum()) + EPS
                    pm1 = pm / pm.sum()
                    Gammajh[h, j] = Gstar[rng.choice(M, p=pm1)]

        # PASO 12 — psi_jh (ancho de banda; Normal truncada positiva)
        for h in range(N - 1):
            mask_h = (Si >= h)
            kh = mask_h.sum()
            for j in range(p):
                if gammajh[h, j] == 0:
                    psijh[h, j] = 0.0
                elif kh > 0:
                    od = (np.sum(psijh[h, :j]  * np.abs(Xnoint[mask_h, :j]  - Gammajh[h, :j]),  axis=1) +
                          np.sum(psijh[h, j+1:] * np.abs(Xnoint[mask_h, j+1:] - Gammajh[h, j+1:]), axis=1))
                    Tijh   = alphah[h] - Zil[mask_h, h] - od
                    dist_j = np.abs(Xnoint[mask_h, j] - Gammajh[h, j])
                    v_psi  = 1.0 / (taupsij[j] + dist_j @ dist_j)
                    m_psi  = v_psi * (taupsij[j] * mupsij[j] + np.sum(Tijh * dist_j))
                    psijh[h, j] = truncnorm_upper0_sample(m_psi, v_psi, rng)

        # PASO 13 — gamma_jh (selección de variables; factor de Bayes analítico)
        for h in range(N):
            for j in range(p):
                act_noj = gammajh[h].copy(); act_noj[j] = 0
                full_noj = np.concatenate([[True], act_noj == 1])
                Xh_noj   = X[:, full_noj]
                Xh1      = np.hstack([Xnoint[:, j:j+1], Xh_noj])
                sb       = (n / g) * np.linalg.inv(Xh1.T @ Xh1) / tauh[h, 0]
                bvec     = np.concatenate([[beta0h[h, 0]], betajh[h]])
                betagh2  = bvec[full_noj]

                if sb.shape[0] > 1:
                    sb11  = sb[0, 0]; sb12 = sb[0, 1:]; sb22 = sb[1:, 1:]
                    sbj   = sb11 - sb12 @ np.linalg.solve(sb22, sb12)
                    taubj = 1.0 / sbj
                    mubj  = sb12 @ np.linalg.solve(sb22, betagh2)
                else:
                    taubj = 1.0 / sb[0, 0]; mubj = 0.0

                idx_h = (Si == h); n_h = idx_h.sum()
                ystar = (y[idx_h]
                         - X[idx_h, 0] * beta0h[h, 0]
                         - Xnoint[idx_h, :j]   @ betajh[h, :j]
                         - Xnoint[idx_h, j+1:] @ betajh[h, j+1:])

                pp = tauh[h, 0] * (Xnoint[idx_h, j] @ Xnoint[idx_h, j]) + taubj
                pm_val = ((tauh[h, 0] * Xnoint[idx_h, j] @ ystar + taubj * mubj) / pp
                          if pp > 0 else 0.0)
                mb = ((norm.logpdf(0, mubj, 1 / np.sqrt(taubj))
                       - norm.logpdf(0, pm_val, np.sqrt(1 / pp)))
                      if pp > 0 else 0.0)

                if h < N - 1:
                    mask_h = (Si >= h); kh = mask_h.sum()
                    od = (np.sum(psijh[h, :j]  * np.abs(Xnoint[mask_h, :j]  - Gammajh[h, :j]),  axis=1) +
                          np.sum(psijh[h, j+1:] * np.abs(Xnoint[mask_h, j+1:] - Gammajh[h, j+1:]), axis=1))
                    Tijh   = alphah[h] - Zil[mask_h, h] - od
                    dist_j = np.abs(Xnoint[mask_h, j] - Gammajh[h, j])
                    v_psi  = 1.0 / (taupsij[j] + dist_j @ dist_j)
                    m_psi  = v_psi * (taupsij[j] * mupsij[j] + np.sum(Tijh * dist_j))

                    lik0_y = norm.logpdf(y[idx_h],
                                         X[idx_h, 0] * beta0h[h, 0] + Xnoint[idx_h, :j] @ betajh[h, :j] + Xnoint[idx_h, j+1:] @ betajh[h, j+1:],
                                         1 / np.sqrt(tauh[h, 0])).sum()
                    lik0_Z = norm.logpdf(Zil[mask_h, h], alphah[h] - od, 1.0).sum()
                    bjhin  = np.log(1 - pij[j] + EPS) + lik0_y + lik0_Z

                    prior_psi = (norm.logpdf(0, mupsij[j], 1 / np.sqrt(taupsij[j]))
                                 - np.log(1 - norm.cdf((0 - mupsij[j]) * np.sqrt(taupsij[j])) + EPS))
                    post_psi  = (-norm.logpdf(0, m_psi, np.sqrt(v_psi))
                                 + np.log(1 - norm.cdf((0 - m_psi) / np.sqrt(v_psi)) + EPS))
                    ajhin = (np.log(pij[j] + EPS)
                             + norm.logpdf(ystar, 0, 1 / np.sqrt(tauh[h, 0])).sum()
                             + mb + norm.logpdf(Tijh, 0, 1.0).sum()
                             + prior_psi + post_psi)

                    prob1 = np.clip(1.0 / (1.0 + np.exp(bjhin - ajhin)), 0.0, 1.0)
                    gammajh[h, j] = rng.binomial(1, prob1)

                else:  # h == N-1
                    b0 = np.exp(1.2 * n_h + np.log(1 - pij[j] + EPS)
                                + norm.logpdf(y[idx_h],
                                              X[idx_h, 0] * beta0h[h, 0] + Xnoint[idx_h, :j] @ betajh[h, :j] + Xnoint[idx_h, j+1:] @ betajh[h, j+1:],
                                              1 / np.sqrt(tauh[h, 0])).sum()) + EPS
                    a0 = np.exp(1.2 * n_h + np.log(pij[j] + EPS)
                                + norm.logpdf(ystar, 0, 1 / np.sqrt(tauh[h, 0])).sum()
                                + mb) + EPS
                    gammajh[h, j] = rng.binomial(1, np.clip(a0 / (a0 + b0), 0.0, 1.0))

        # PASO 14 — Almacenar trazas
        max_si = int(Si.max())
        osumout[gt]      = (gammajh[:max_si + 1].sum(axis=0) == 0).astype(float)
        muout[gt]        = mu
        tauhout[gt]      = tauh[:, 0]
        beta0hout[gt]    = beta0h[:, 0]
        betajhout[gt]    = betajh
        alphahout[gt]    = alphah
        psijhout[gt]     = psijh
        Gammajhout[gt]   = Gammajh
        gammajhout[gt]   = gammajh
        pijout[gt]       = pij
        wjout[gt]        = wj
        N1out[gt]        = max_si + 1
        Nout[gt]         = N
        inEout[gt]       = inE.sum(axis=1)
        outEout[gt]      = outE.sum(axis=1)

        if (gt + 1) % 200 == 0:
            print(f"  [{EXPERIMENT_ID}] iter {gt + 1}/{nsim}")

    # ── 6. Post-procesamiento ────────────────────────────────────────────────
    post = slice(burn, nsim)
    gloprob = osumout[post].mean(axis=0)
    inPred  = np.mean(y1) + np.std(y1) * inEout[post].mean(axis=0)
    outPred = np.mean(y1) + np.std(y1) * outEout[post].mean(axis=0)
    inRMSE  = np.sqrt(np.mean((inPred  - y1) ** 2))
    outRMSE = np.sqrt(np.mean((outPred - y2) ** 2))

    return dict(
        # trazas MCMC
        betajhout=betajhout, beta0hout=beta0hout, tauhout=tauhout,
        alphahout=alphahout, psijhout=psijhout, Gammajhout=Gammajhout,
        gammajhout=gammajhout, pijout=pijout, wjout=wjout,
        muout=muout, osumout=osumout, N1out=N1out, Nout=Nout,
        inEout=inEout, outEout=outEout,
        # datos
        X=X, outX=outX, Xnoint=Xnoint,
        y=y, y1=y1, y2=y2, outy=outy,
        # métricas y predicciones
        inPred=inPred, outPred=outPred,
        inRMSE=float(inRMSE), outRMSE=float(outRMSE),
        gloprob=gloprob,
        # config
        nsim=nsim, burn=burn, p=p, n=n,
    )

## 3. Ejecución del modelo

In [ ]:
# Ambos archivos se leen desde raw/ — son inputs del modelo, no outputs
PATH_IN  = PATHS["raw"] / f"{BASEFNAME}in_{TT}.txt"
PATH_OUT = PATHS["raw"] / f"{BASEFNAME}out_{TT}.txt"

if not PATH_IN.exists():
    raise FileNotFoundError(f"Train no encontrado: {PATH_IN}")
if not PATH_OUT.exists():
    raise FileNotFoundError(f"Test  no encontrado: {PATH_OUT}")

print(f"  train → {PATH_IN.relative_to(PROJECT_ROOT)}")
print(f"  test  → {PATH_OUT.relative_to(PROJECT_ROOT)}")
print(f"\nIniciando MCMC [{EXPERIMENT_ID}] — {MCMC_CONFIG['nsim']} iteraciones ...\n")

results = run_psbp(
    path_in  = PATH_IN,
    path_out = PATH_OUT,
    mcmc_cfg = MCMC_CONFIG,
    hp       = HYPERPARAMS,
    seed     = SEED,
)

print(f"\n{'='*55}")
print(f"  in-sample  RMSE : {results['inRMSE']:.6f}")
print(f"  out-sample RMSE : {results['outRMSE']:.6f}")
print(f"  Inclusión global (1-gloprob): {(1 - results['gloprob']).round(3).tolist()}")
print(f"{'='*55}")

  train → data\simulaciones\raw\BHPin_1.txt
  test  → data\simulaciones\raw\BHPout_1.txt

Iniciando MCMC [psbp_sim_BHP_1_20260509_203336] — 2000 iteraciones ...



C:\Users\JuanFran\AppData\Local\Temp\ipykernel_3216\2767828171.py:335: RuntimeWarning: overflow encountered in exp
  prob1 = np.clip(1.0 / (1.0 + np.exp(bjhin - ajhin)), 0.0, 1.0)


  [psbp_sim_BHP_1_20260509_203336] iter 200/2000


## 4. Persistencia de artefactos

Se guardan **cuatro artefactos** en sus rutas canónicas:

| Artefacto | Ruta | Contenido |
|-----------|------|-----------|
| `_traces.npz` | `artefact/simulaciones/models/` | Trazas MCMC comprimidas |
| `_results.pkl` | `artefact/simulaciones/models/` | Dict `results` completo |
| `_meta.json` | `artefact/simulaciones/models/` | Metadata + métricas (legible) |
| `_predictions.npz` | `data/simulaciones/processed/predict/` | `inPred`, `outPred`, `y1`, `y2` |

In [ ]:
# ── 4.1 Trazas MCMC → .npz ───────────────────────────────────────────────────
traces_keys = [
    "betajhout", "beta0hout", "tauhout", "alphahout",
    "psijhout", "Gammajhout", "gammajhout", "pijout",
    "wjout", "muout", "osumout", "N1out", "Nout",
    "inEout", "outEout",
]
path_npz = PATHS["artifact"] / f"{EXPERIMENT_ID}_traces.npz"
np.savez_compressed(path_npz, **{k: results[k] for k in traces_keys})
print(f"[OK] Trazas MCMC  → {path_npz}")

# ── 4.2 Objeto results completo → .pkl ───────────────────────────────────────
path_pkl = PATHS["artifact"] / f"{EXPERIMENT_ID}_results.pkl"
with open(path_pkl, "wb") as f:
    pickle.dump(results, f, protocol=pickle.HIGHEST_PROTOCOL)
print(f"[OK] Results pkl  → {path_pkl}")

# ── 4.3 Metadata + métricas → .json ─────────────────────────────────────────
meta = {
    "experiment_id": EXPERIMENT_ID,
    "timestamp":     TIMESTAMP,
    "seed":          SEED,
    "basefname":     BASEFNAME,
    "tt":            TT,
    "data_type":     "simulaciones",
    "path_in":       str(PATH_IN),
    "path_out":      str(PATH_OUT),
    "mcmc_config":   MCMC_CONFIG,
    "hyperparams":   HYPERPARAMS,
    "metrics": {
        "inRMSE":   results["inRMSE"],
        "outRMSE":  results["outRMSE"],
        "gloprob":  results["gloprob"].tolist(),
        "inclusion_prob": (1 - results["gloprob"]).tolist(),
    },
    "artifacts": {
        "traces_npz": str(path_npz),
        "results_pkl": str(path_pkl),
    }
}
path_json = PATHS["artifact"] / f"{EXPERIMENT_ID}_meta.json"
with open(path_json, "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)
print(f"[OK] Metadata     → {path_json}")

# ── 4.4 Predicciones → data/simulaciones/processed/predict/ ─────────────────
path_pred = PATHS["predict"] / f"{EXPERIMENT_ID}_predictions.npz"
np.savez_compressed(
    path_pred,
    inPred=results["inPred"],
    outPred=results["outPred"],
    y1=results["y1"],
    y2=results["y2"],
)
print(f"[OK] Predicciones → {path_pred}")

## 5. Diagnósticos rápidos post-MCMC

Verificaciones mínimas de convergencia y calidad del ajuste.

In [ ]:
import matplotlib.pyplot as plt

burn   = results["burn"]
nsim   = results["nsim"]
post   = slice(burn, nsim)

fig, axes = plt.subplots(2, 2, figsize=(12, 7))
fig.suptitle(f"Diagnósticos MCMC — {EXPERIMENT_ID}", fontsize=11)

# (a) Número de componentes activos
ax = axes[0, 0]
ax.plot(results["N1out"], color="steelblue", lw=0.7, alpha=0.8)
ax.axvline(burn, color="crimson", lw=1.2, ls="--", label=f"burn-in ({burn})")
ax.set_title("Componentes activos $N_1^{(t)}$")
ax.set_xlabel("Iteración"); ax.set_ylabel("$N_1$")
ax.legend(fontsize=8)

# (b) Traza de mu
ax = axes[0, 1]
ax.plot(results["muout"], color="darkorange", lw=0.7, alpha=0.8)
ax.axvline(burn, color="crimson", lw=1.2, ls="--")
ax.set_title("Traza de $\\mu$")
ax.set_xlabel("Iteración"); ax.set_ylabel("$\\mu$")

# (c) Predicciones in-sample vs valores reales
ax = axes[1, 0]
y1    = results["y1"]
inPred = results["inPred"]
ax.scatter(y1, inPred, s=10, alpha=0.5, color="steelblue")
lims = [min(y1.min(), inPred.min()), max(y1.max(), inPred.max())]
ax.plot(lims, lims, "k--", lw=1)
ax.set_title(f"In-sample: RMSE = {results['inRMSE']:.4f}")
ax.set_xlabel("$y$ observado"); ax.set_ylabel("$\\hat{y}$ predicho")

# (d) Predicciones out-of-sample vs valores reales
ax = axes[1, 1]
y2     = results["y2"]
outPred = results["outPred"]
ax.scatter(y2, outPred, s=10, alpha=0.5, color="darkorange")
lims = [min(y2.min(), outPred.min()), max(y2.max(), outPred.max())]
ax.plot(lims, lims, "k--", lw=1)
ax.set_title(f"Out-of-sample: RMSE = {results['outRMSE']:.4f}")
ax.set_xlabel("$y$ observado"); ax.set_ylabel("$\\hat{y}$ predicho")

plt.tight_layout()

# Guardar figura en reports/
path_fig = PATHS["reports"] / f"{EXPERIMENT_ID}_diagnostics.png"
fig.savefig(path_fig, dpi=150, bbox_inches="tight")
print(f"[OK] Figura guardada → {path_fig}")
plt.show()

In [ ]:
# ── Probabilidades de inclusión por variable ──────────────────────────────────
p = results["p"]
incl = 1 - results["gloprob"]

fig, ax = plt.subplots(figsize=(max(5, p * 0.8), 4))
ax.bar(range(p), incl, color="steelblue", edgecolor="white")
ax.axhline(0.5, color="crimson", lw=1.2, ls="--", label="umbral 0.5")
ax.set_xticks(range(p))
ax.set_xticklabels([f"$x_{{{j+1}}}$" for j in range(p)])
ax.set_ylim(0, 1.05)
ax.set_title("Probabilidad de inclusión global por variable")
ax.set_ylabel("$P(\\gamma_j = 1 \\mid \\text{data})$")
ax.legend(fontsize=9)

path_incl = PATHS["reports"] / f"{EXPERIMENT_ID}_inclusion_probs.png"
fig.savefig(path_incl, dpi=150, bbox_inches="tight")
print(f"[OK] Inclusión  → {path_incl}")
plt.show()

## 6. Resumen del experimento

In [ ]:
print("=" * 60)
print(f"  RESUMEN EXPERIMENTO: {EXPERIMENT_ID}")
print("=" * 60)
print(f"  Datos train  : {PATH_IN.name}  (n = {results['n']})")
print(f"  Datos test   : {PATH_OUT.name} (n2 = {len(results['y2'])})")
print(f"  Covariables  : p = {results['p']}")
print(f"  MCMC         : {MCMC_CONFIG['nsim']} iter, burn-in = {MCMC_CONFIG['burn']}")
print(f"  N trunc.     : {MCMC_CONFIG['N']}    Grilla M : {MCMC_CONFIG['M']}")
print(f"  Seed         : {SEED}")
print("-" * 60)
print(f"  inRMSE       : {results['inRMSE']:.6f}")
print(f"  outRMSE      : {results['outRMSE']:.6f}")
print("-" * 60)
print("  Artefactos guardados:")
print(f"    {path_npz.relative_to(PROJECT_ROOT)}")
print(f"    {path_pkl.relative_to(PROJECT_ROOT)}")
print(f"    {path_json.relative_to(PROJECT_ROOT)}")
print(f"    {path_pred.relative_to(PROJECT_ROOT)}")
print("  Reportes:")
print(f"    {path_fig.relative_to(PROJECT_ROOT)}")
print(f"    {path_incl.relative_to(PROJECT_ROOT)}")
print("=" * 60)